UNet - perception in v2 - SR

In [ ]:
from fastai.vision.all import *
import cv2
import numpy
import os
import pathlib
import torch
import fastai; fastai.__version__

In [ ]:
torch.cuda.set_device(0)
torch.cuda.current_device(), torch.cuda.get_device_name(0)

In [ ]:
path = Path.cwd()/'data'

In [ ]:
path_LR = path
subfolders = ['x3']

In [ ]:
# Set seeds for reproducibility
%run util/set_seeds.ipynb

# run Dataloader
%run util/Diffusion_Dataloader_train.ipynb

# run notebook to set up feature loss 
%run util/Diffusion_Perception_loss.ipynb

In [ ]:
dls_den, dblock = get_dls(bs=32) # test with smaller batch size for now

In [ ]:
dblock.summary(path_LR) # check pipelines are working as expected 

In [ ]:
# dummy definition of feature loss function using the VGG model and specified blocks and weights
loss_func = FeatureLoss(vgg_m, blocks,[125], [1,1,1,1,1], [1,1,1,1,1])

In [ ]:
def create_gen_learner_others():
    return unet_learner(dls_den, bbone, loss_func=loss_func, blur=True, norm_type=NormType.Weight, 
                        self_attention=True, y_range=y_range, pretrained=True, 
                        metrics=metrics, cbs=cbs
                        )

In [ ]:
#training loop for different feature loss configurations, first and second layers
inputs = ['x3']
losses = ['Feat_0,','Feat_1','baseline']
for j in inputs:
    path_LR = path/j        
    dls_den, dblock = get_dls(bs=120) # check if your GPU can handle this batch size, otherwise reduce it
    metrics = LossMetrics(loss_func.metric_names)
    for i in losses:
        if '0' in i:
            fname='SR_R18_x3_Feat0'
            cbs=[CSVLogger(fname=fname+'.csv')]
            loss_func = FeatureLoss(vgg_m, blocks,[125], [92,0,0,0,0], [0,0,0,0,0])
            learn_den = create_gen_learner_others()
            learn_den.load('SR_R18_Feat0')
        if '1' in i:
            fname='SR_R18_x3_Feat1'
            cbs=[CSVLogger(fname=fname+'.csv')]
            loss_func = FeatureLoss(vgg_m, blocks,[125], [0,52,0,0,0], [0,0,0,0,0])
            learn_den = create_gen_learner_others()
            learn_den.load('SR_R18_Feat1')
        if 'baseline' in i:
            fname='SR_R18_x3_baseline'
            cbs=[CSVLogger(fname=fname+'.csv')]
            loss_func = FeatureLoss(vgg_m, blocks,[125], [0,0,0,0,0], [0,0,0,0,0])
            learn_den = create_gen_learner_others()
            learn_den.load('SR_R18_L1')
        learn_den.fine_tune(20,2e-4,wd=wd,freeze_epochs=5)
        learn_den.save(fname)